# NB 02: FT-Transformer Hyperparameter Sweep

Grid search over:
- **Learning rate:** 1e-3, 3e-4, 1e-4, 3e-5
- **n_layers:** 2, 3, 4
- **d_token:** 128, 192, 256

36 configurations, early-stopped on val AUROC. Select best config by
average test AUROC across all evaluable outcomes.

**Expected runtime:** ~10-20 min per config on T4 -> ~6-12 hours total for 36 configs.

**Prerequisites:**
1. Run `00_setup_and_preprocessing.ipynb` first (saves artifact to Drive)
2. Runtime -> Change runtime type -> **T4 GPU** (or A100)
3. GitHub PAT stored in Colab Secrets as `GITHUB_PAT`

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
import os

# Retrieve the GitHub token from Colab Secrets
github_token = userdata.get('GITHUB_PAT')

username = "elroy-galbraith"
repository = "fin-jepa"
repo_url = f"https://{github_token}@github.com/{username}/{repository}.git"

if not os.path.exists(f'/content/{repository}'):
    !git clone {repo_url} /content/{repository}

%cd /content/{repository}

# Install fin-jepa as editable WITHOUT upgrading Colab's pre-installed deps
!pip install -q --no-deps -e '.[dev]'

# Install only the deps Colab doesn't already ship
!pip install -q hydra-core omegaconf mlflow optuna xgboost yfinance rich python-dotenv tqdm

In [ ]:
# Symlink data from Google Drive so relative paths work
import os

DRIVE_DATA = '/content/drive/MyDrive/Fin-JEPA/data'

for subdir in ['raw', 'processed']:
    target = f'data/{subdir}'
    if os.path.islink(target) or os.path.isdir(target):
        !rm -rf {target}
    os.symlink(f'{DRIVE_DATA}/{subdir}', target)

# Verify data exists
!ls -la data/raw/xbrl_features.parquet
!ls -la data/processed/label_database.parquet
!ls -la data/raw/market/market_aligned.parquet
print('Data linked successfully!')

# Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU detected! Change runtime to T4 or A100.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
props = torch.cuda.get_device_properties(0)
print(f'VRAM: {props.total_memory / 1e9:.1f} GB')

In [ ]:
# Common setup: logging, configs, control flag
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from omegaconf import OmegaConf

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# Set to True to re-run experiments even if cached results exist
FORCE_RERUN = False

# Load configs from YAML (override individual values below if needed)
benchmark_cfg = OmegaConf.load('configs/study0/benchmark.yaml')
ssl_cfg = OmegaConf.load('configs/study0/ssl_experiment.yaml')

# Paths
RESULTS_DIR = Path('results/study0')
BENCHMARK_PATH = RESULTS_DIR / 'benchmark_results.json'
SWEEP_PATH = RESULTS_DIR / 'ft_sweep_results.json'
SSL_V2_PATH = RESULTS_DIR / 'ssl_experiment_results_v2.json'
FINAL_PATH = RESULTS_DIR / 'final_benchmark.json'

OUTCOMES = ['stock_decline', 'earnings_restate', 'audit_qualification',
            'sec_enforcement', 'bankruptcy']

print(f'Results dir: {RESULTS_DIR}')
print(f'Benchmark cached: {BENCHMARK_PATH.exists()}')
print(f'Sweep cached: {SWEEP_PATH.exists()}')
print(f'SSL v2 cached: {SSL_V2_PATH.exists()}')
print(f'FORCE_RERUN: {FORCE_RERUN}')

# Preprocessing artifact path (shared across split notebooks)
ARTIFACT_DIR = Path('/content/drive/MyDrive/Fin-JEPA/artifacts/study0')
ARTIFACT_PATH = ARTIFACT_DIR / 'preprocessing_v1.pkl'

SEED = int(OmegaConf.select(benchmark_cfg, 'training.seed', default=42))

mask_ratios = [0.15, 0.30, 0.50]
MIN_POSITIVES = 20


In [ ]:
import pickle

if not ARTIFACT_PATH.exists():
    raise FileNotFoundError(
        f'Preprocessing artifact not found at {ARTIFACT_PATH}.\n'
        'Run 00_setup_and_preprocessing.ipynb first.'
    )

with open(ARTIFACT_PATH, 'rb') as f:
    _artifact = pickle.load(f)

splits = _artifact['splits']
scaler = _artifact['scaler']
feature_cols = _artifact['feature_cols']
categorical_cols = _artifact['categorical_cols']
n_features = _artifact['n_features']
n_cat = _artifact['n_cat']
cat_cards = _artifact['cat_cards']
config_fingerprint = _artifact['config_fingerprint']

print(f'Loaded preprocessing artifact (fingerprint: {config_fingerprint})')
print(f'  Train: {len(splits["train"]):,} | Val: {len(splits["val"]):,} | Test: {len(splits["test"]):,}')
print(f'  Features: {n_features} ({n_cat} categorical)')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
%%time
from itertools import product
from fin_jepa.utils.reproducibility import seed_everything
from fin_jepa.models.ft_transformer import FTTransformer
from fin_jepa.training.dataset import make_dataloader
from fin_jepa.training.train_study0 import train_ft_transformer, _predict_scores, MIN_POSITIVES
from fin_jepa.training.metrics import compute_all_metrics
import torch.nn as nn

# Sweep grid
SWEEP_LRS = [1e-3, 3e-4, 1e-4, 3e-5]
SWEEP_LAYERS = [2, 3, 4]
SWEEP_DTOKENS = [128, 192, 256]

total = len(SWEEP_LRS) * len(SWEEP_LAYERS) * len(SWEEP_DTOKENS)
print(f'Grid: {len(SWEEP_LRS)} lr x {len(SWEEP_LAYERS)} layers x {len(SWEEP_DTOKENS)} d_token = {total} configs')

if not FORCE_RERUN and SWEEP_PATH.exists():
    print(f'Loading cached sweep results from {SWEEP_PATH}')
    with open(SWEEP_PATH) as f:
        sweep_results = json.load(f)
else:
    sweep_results = {'configs': [], 'best_config': None}
    batch_size = 256
    _cat_cols = categorical_cols or None

    for i, (lr, n_layers, d_token) in enumerate(product(SWEEP_LRS, SWEEP_LAYERS, SWEEP_DTOKENS)):
        # n_heads must divide d_token
        n_heads = 8 if d_token % 8 == 0 else 4
        config_name = f'lr={lr:.0e}_layers={n_layers}_d={d_token}'
        print(f'\n[{i+1}/{total}] {config_name}')

        model_kwargs = {
            'n_features': n_features,
            'd_token': d_token,
            'n_heads': n_heads,
            'n_layers': n_layers,
            'd_ffn_factor': 4,
            'dropout': 0.0,
            'n_outputs': 1,
            'n_cat_features': n_cat,
            'cat_cardinalities': cat_cards,
        }

        config_result = {
            'lr': lr, 'n_layers': n_layers, 'd_token': d_token, 'n_heads': n_heads,
            'outcomes': {},
        }

        for outcome in OUTCOMES:
            train_df = splits['train'][splits['train'][outcome].notna()]
            val_df = splits['val'][splits['val'][outcome].notna()]
            test_df = splits['test'][splits['test'][outcome].notna()]
            n_pos = int(train_df[outcome].sum())
            if n_pos < MIN_POSITIVES:
                config_result['outcomes'][outcome] = {'auroc': None, 'skipped': True}
                print(f'  {outcome}: skipped')
                continue

            n_neg = len(train_df) - n_pos
            pos_weight = n_neg / max(n_pos, 1)

            train_loader = make_dataloader(train_df, feature_cols, outcome, batch_size, shuffle=True, cat_feature_cols=_cat_cols, seed=SEED)
            val_loader = make_dataloader(val_df, feature_cols, outcome, batch_size, shuffle=False, cat_feature_cols=_cat_cols)
            test_loader = make_dataloader(test_df, feature_cols, outcome, batch_size, shuffle=False, cat_feature_cols=_cat_cols)

            seed_everything(SEED)
            model = FTTransformer(**model_kwargs).to(device)
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
            criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))

            train_ft_transformer(model, train_loader, val_loader, criterion, optimizer,
                                 device, epochs=100, patience=10)

            y_true, y_score = _predict_scores(model, test_loader, device)
            metrics = compute_all_metrics(y_true, y_score)
            config_result['outcomes'][outcome] = metrics
            auroc = metrics.get('auroc')
            print(f'  {outcome}: {auroc:.4f}' if auroc else f'  {outcome}: skipped')

        # Average test AUROC across evaluable outcomes
        aurocs = [m['auroc'] for m in config_result['outcomes'].values()
                  if m.get('auroc') is not None]
        config_result['mean_auroc'] = float(np.mean(aurocs)) if aurocs else 0.0
        sweep_results['configs'].append(config_result)

    # Select best config by mean AUROC
    best = max(sweep_results['configs'], key=lambda c: c['mean_auroc'])
    sweep_results['best_config'] = {
        'lr': best['lr'], 'n_layers': best['n_layers'],
        'd_token': best['d_token'], 'n_heads': best['n_heads'],
        'mean_auroc': best['mean_auroc'],
    }

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(SWEEP_PATH, 'w') as f:
        json.dump(sweep_results, f, indent=2, default=str)
    print(f'\nSweep results saved to {SWEEP_PATH}')

best_cfg = sweep_results['best_config']
print(f"\nBest config: lr={best_cfg['lr']}, n_layers={best_cfg['n_layers']}, "
      f"d_token={best_cfg['d_token']}, mean_auroc={best_cfg['mean_auroc']:.4f}")

In [ ]:
# Sweep results: top 10 configs by mean AUROC
sweep_rows = []
for cfg in sweep_results['configs']:
    row = {'lr': cfg['lr'], 'n_layers': cfg['n_layers'], 'd_token': cfg['d_token'],
           'mean_auroc': cfg['mean_auroc']}
    for outcome in OUTCOMES:
        row[outcome] = cfg['outcomes'].get(outcome, {}).get('auroc')
    sweep_rows.append(row)

sweep_df = pd.DataFrame(sweep_rows).sort_values('mean_auroc', ascending=False)
sweep_df.head(10).style.format({'lr': '{:.0e}', 'mean_auroc': '{:.4f}',
    **{o: '{:.4f}' for o in OUTCOMES}}, na_rep='--').background_gradient(
    subset=['mean_auroc'], cmap='Greens')

In [ ]:
# Heatmap: mean AUROC by lr x (n_layers, d_token)
pivot_rows = []
for cfg in sweep_results['configs']:
    pivot_rows.append({
        'lr': f"{cfg['lr']:.0e}",
        'arch': f"L={cfg['n_layers']} D={cfg['d_token']}",
        'mean_auroc': cfg['mean_auroc'],
    })

pivot_df = pd.DataFrame(pivot_rows).pivot(index='lr', columns='arch', values='mean_auroc')

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot_df, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Mean Test AUROC'})
ax.set_title('FT-Transformer Sweep: Mean AUROC by LR x Architecture')
ax.set_ylabel('Learning Rate')
ax.set_xlabel('Architecture (Layers, d_token)')
plt.tight_layout()
fig_dir = RESULTS_DIR / 'figures'
fig_dir.mkdir(exist_ok=True)
plt.savefig(fig_dir / 'sweep_heatmap.png', bbox_inches='tight', dpi=300)
plt.show()
